# ChemiAI — финальный воспроизводимый best-пайплайн

Этот ноутбук собирает текущий лучший публичный сабмит:

`submission_transductive_neighbor_target_blend.csv` -> **298.48843**

Финальная формула:

```text
IC50 = clustering baseline
CC50 = 40% clustering baseline + 60% transductive neighbor
SI   = 70% clustering baseline + 30% transductive neighbor
```

Важно: `submission_clustering.csv` используется как уже подтвержденный clustering baseline из `solution_clustering.ipynb`.

## Почему пайплайн устроен так

Лучший чистый clustering baseline (`submission_clustering.csv`) дал `306.26468`. Основной последующий прирост пришел от transductive neighbor-модели, но только для `CC50` и частично для `SI`. `IC50` у neighbor-модели локально хуже, поэтому `IC50` оставляем из clustering baseline.

Transductive preprocessing означает, что `SimpleImputer`, `StandardScaler` и `PCA` обучаются на `train + test` без таргетов. Это не использует ответы теста, но лучше подстраивает геометрию соседей под реальные test-молекулы.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TARGET_COLS = ["IC50, mM", "CC50, mM", "SI"]
SUBMISSION_COLS = ["IC50", "CC50", "SI"]
ID_COL = "index"

DATA_DIR = Path("data")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"
CLUSTERING_BASELINE_PATH = Path("submission_clustering.csv")
BEST_SUBMISSION_PATH = Path("submission_transductive_neighbor_target_blend.csv")

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

feature_cols = [c for c in train.columns if c not in [ID_COL, *TARGET_COLS]]
X_train = train[feature_cols].copy()
y_train = train[TARGET_COLS].copy()
X_test = test[feature_cols].copy()

const_cols = [c for c in X_train.columns if X_train[c].nunique(dropna=False) <= 1]
X_train = X_train.drop(columns=const_cols)
X_test = X_test.drop(columns=const_cols)

print("Train:", train.shape)
print("Test:", test.shape)
print("Features after constant drop:", X_train.shape[1])
print("Dropped constant columns:", len(const_cols))

In [ ]:
def competition_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    rmses = [np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])) for i in range(y_true.shape[1])]
    return float(np.mean(rmses))


def target_rmse(y_true: np.ndarray, y_pred: np.ndarray) -> list[float]:
    return [float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))) for i in range(y_true.shape[1])]


def make_transductive_pca_features(
    X_fit_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    X_test_full: pd.DataFrame,
    n_components: int = 20,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Fit unsupervised preprocessing on train-fold + test, then transform train/valid/test."""
    fit_frame = pd.concat([X_fit_train, X_test_full], axis=0)

    imputer = SimpleImputer(strategy="median")
    fit_imputed = imputer.fit_transform(fit_frame)

    scaler = StandardScaler()
    fit_scaled = scaler.fit_transform(fit_imputed)

    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    pca.fit(fit_scaled)

    X_fit_pca = pca.transform(scaler.transform(imputer.transform(X_fit_train)))
    X_valid_pca = pca.transform(scaler.transform(imputer.transform(X_valid)))
    X_test_pca = pca.transform(scaler.transform(imputer.transform(X_test_full)))

    return X_fit_pca, X_valid_pca, X_test_pca

## Transductive neighbor-модель

Параметры финальной neighbor-модели:

```text
PCA(n_components=20) + KNN(k=5, weights="distance")
```

Модель обучается в `log1p`-шкале по всем трем таргетам. В финальном сабмите используются только `CC50` и `SI` из этой модели.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

transductive_oof = np.zeros((len(X_train), len(TARGET_COLS)), dtype=float)
transductive_test_pred = np.zeros((len(X_test), len(TARGET_COLS)), dtype=float)

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train), start=1):
    X_fit = X_train.iloc[train_idx]
    X_valid = X_train.iloc[valid_idx]
    y_fit = y_train.iloc[train_idx].values

    X_fit_pca, X_valid_pca, X_test_pca = make_transductive_pca_features(
        X_fit_train=X_fit,
        X_valid=X_valid,
        X_test_full=X_test,
        n_components=20,
    )

    model = KNeighborsRegressor(n_neighbors=5, weights="distance")
    model.fit(X_fit_pca, np.log1p(y_fit))

    valid_pred = np.expm1(model.predict(X_valid_pca))
    test_pred = np.expm1(model.predict(X_test_pca))

    transductive_oof[valid_idx] = np.clip(valid_pred, 0, None)
    transductive_test_pred += np.clip(test_pred, 0, None) / kf.n_splits

print("Transductive neighbor OOF:", competition_score(y_train.values, transductive_oof))
print("Per-target RMSE:", np.round(target_rmse(y_train.values, transductive_oof), 5).tolist())

## Сборка финального сабмита

Берем `submission_clustering.csv` как проверенный public baseline и смешиваем его с transductive neighbor-предсказаниями только по `CC50` и `SI`.

`IC50` не трогаем, потому что neighbor-модель ухудшала этот таргет локально.

In [ ]:
if not CLUSTERING_BASELINE_PATH.exists():
    raise FileNotFoundError(
        "Нужен submission_clustering.csv — подтвержденный clustering baseline из solution_clustering.ipynb"
    )

clustering_baseline = pd.read_csv(CLUSTERING_BASELINE_PATH)
assert list(clustering_baseline.columns) == ["index", *SUBMISSION_COLS]
assert list(sample_submission.columns) == ["index", *SUBMISSION_COLS]
assert clustering_baseline.shape == sample_submission.shape

submission_best = clustering_baseline.copy()
submission_best["IC50"] = clustering_baseline["IC50"].values
submission_best["CC50"] = (
    0.40 * clustering_baseline["CC50"].values
    + 0.60 * transductive_test_pred[:, TARGET_COLS.index("CC50, mM")]
)
submission_best["SI"] = (
    0.70 * clustering_baseline["SI"].values
    + 0.30 * transductive_test_pred[:, TARGET_COLS.index("SI")]
)

submission_best[SUBMISSION_COLS] = submission_best[SUBMISSION_COLS].clip(lower=0)
submission_best.to_csv(BEST_SUBMISSION_PATH, index=False)

print("Saved:", BEST_SUBMISSION_PATH)
print(submission_best.head())
print(submission_best.describe())

In [ ]:
# Финальные проверки формата сабмита.
submission_check = pd.read_csv(BEST_SUBMISSION_PATH)

assert list(submission_check.columns) == ["index", *SUBMISSION_COLS]
assert submission_check.shape == sample_submission.shape
assert submission_check[SUBMISSION_COLS].notna().all().all()
assert (submission_check[SUBMISSION_COLS] >= 0).all().all()
assert submission_check["index"].equals(sample_submission["index"])

print("Submission format is valid")
print("Rows:", len(submission_check))
print("Columns:", submission_check.columns.tolist())